## **传统回归模型**

本章内容虽标题是回归，但只有线性回归是真正解决回归问题，逻辑回归和softmax回归分别用于解决二分类和多分类问题。

### **线性回归**
***线性回归***是一种最简单的回归方法，它假设特征和目标变量之间的关系可以用一条**直线**表示。
例如，我们手里有一些房屋信息的数据：

<div align="center">

|   房屋面积   |   50    |   80    |   100   |   120   |   150   |   90    |   110   |   130   |  100   |  140  |
| :----------: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :----: | :---: |
| **房间数量** |  **2**  |  **3**  |  **3**  |  **4**  |  **4**  |  **3**  |  **4**  |  **5**  | **3**  | **4** |
| **房屋年龄** | **10**  |  **8**  | **15**  | **15**  | **20**  | **12**  |  **7**  |  **3**  | **10** | **5** |
|   **房价**   | **180** | **240** | **260** | **300** | **350** | **280** | **320** | **360** | **?**  | **?** |

</div>

#### **核心思想**

试图找到一个最佳的“**直线**”，以便用特征（如面积、房间数、房龄）来预测变量（如房价）。

数学上，这条直线的方程是：$y = w_1 \cdot x_1 + w_2 \cdot x_2 + \dots + w_n \cdot x_n + b = \Sigma (w_i \cdot x_i)+b$​

* $y$ 是我们要预测的目标（房价）。
* $x_1, x_2, ..., x_n$ 是特征（面积、房龄等）。
* $w_1, w_2, ..., w_n$ 是特征对应的权重，描述了每个特征对目标值的影响程度（**是要学习的参数**）。
* $b$ 是偏置，描述了当所有特征为0时，预测值是多少（**同样是需要学习的参数**）。

> 在本例中它的特征变量有三个（面积、房间数、房龄），无法在平面坐标系上绘制出这条直线，但它在高维空间中仍然是以一条直线的形式存在（因为多项式次数为 1）。

在实际应用中，我们更喜欢用**矩阵形式**来表示，即：$\vec{y} = X \cdot \vec{w}$

* 其中，$X$中第 i 行某样本$\vec{x_i} = [x_{i1}, x_{i2}, \dots, x_{in}, 1]$，参数$\vec{w} = [w_{1}, w_{2}, \dots, w_{n}, b]^T$(等价变换，可以把 $b$ 也算到参数 $w$ 里)

#### **损失函数**

在回归任务中，最常用的损失函数是**均方误差（MSE, Mean Squared Error）**
$MSE = {1 \over 2} \sum\limits_{i=1}^m(\hat y_i - y_i)^2$

* $m$ 是样本的数量，$\hat y_i$ 是第 $i$ 个样本的**预测值**，$y_i$ 是第 $i$ 个样本的真实值。

* 该损失函数可以反映出模型在**所有样本上的总体误差**情况


其矩阵形式为：$MSE = {1 \over 2}(\mathbf{\hat y} - \mathbf{y})^T(\mathbf{\hat y} - \mathbf{y}) = {1\over 2}(\mathbf{y} - X\mathbf{w})^T(\mathbf{y} - X\mathbf{w})$

定义损失函数后，问题就转化为求解最优参数 $\mathbf{w}$，使得损失函数最小化的问题。

因为损失函数是一个**凸函数**，有局部最小值等于全局最小值的性质。因此可以用直接求导法和梯度下降法求解。



##### 直接求导法
${\partial J(\mathbf{w}) \over \partial \mathbf{w}} = X^TX \mathbf{w} - X^T \mathbf{y}$

令其为$\mathbf{0}$，即：

$X^TX \mathbf{w} = X^T \mathbf{y}$，

求解得：

$\mathbf{w} = (X^TX)^{-1} X^T\mathbf{y}$

In [3]:
import numpy as np
X = np.array([
    [50, 2, 10, 1],   # 面积50平米，2个房间，10年房龄
    [80, 3, 8, 1],    # 面积80平米，3个房间，8年房龄
    [100, 3, 15, 1],  # 面积100平米，3个房间，15年房龄
    [120, 4, 15, 1],  # 面积120平米，4个房间，15年房龄
    [150, 4, 20, 1],  # 面积150平米，4个房间，20年房龄
    [90, 3, 12, 1],   # 面积90平米，3个房间，12年房龄
    [110, 4, 7, 1],   # 面积110平米，4个房间，7年房龄
    [130, 5, 3, 1]    # 面积130平米，5个房间，3年房龄
])

# 对应的房价（单位：万元）
y = np.array([180, 240, 260, 300, 350, 280, 320, 360])

w = np.linalg.inv(X.T @ X) @ X.T @ y
w


array([  2.41156841, -14.61067853,  -4.37708565, 136.42936596])

用求得的参数列表计算MSE

In [4]:
from sklearn.metrics import mean_squared_error
# 测试数据
X_test = np.array([
    [100, 3, 10, 1],
    [140, 4, 5, 1]
])

# 测试数据的房价
y_test = np.array([290, 395])

# 预测房价
y_pred = X_test @ w
print("预测的房价：", y_pred)

mse = mean_squared_error(y_test, y_pred)
print(f"均方误差: {mse}")

预测的房价： [289.98331479 393.72080089]
均方误差: 0.8183143797833334


##### **梯度下降法**



In [5]:
# 不标准化就会数值溢出

alpha = 0.1
w = [0, 0, 0, 0]
for i in range(1000):
    w = w - alpha * (X.T @ X @ w - X.T @ y)
w

C:\Users\程鲲鹏\AppData\Local\Temp\ipykernel_23732\2104471159.py:6: RuntimeWarning: overflow encountered in matmul
  w = w - alpha * (X.T @ X @ w - X.T @ y)
C:\Users\程鲲鹏\AppData\Local\Temp\ipykernel_23732\2104471159.py:6: RuntimeWarning: invalid value encountered in subtract
  w = w - alpha * (X.T @ X @ w - X.T @ y)


array([nan, nan, nan, nan])

In [6]:
# 标准化
from sklearn.preprocessing import StandardScaler
print(X,y)
# 创建StandardScaler
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_normalized = scaler_X.fit_transform(X)
y_normalized = scaler_y.fit_transform(y.reshape(-1, 1)).reshape(-1)
X_normalized, y_normalized

[[ 50   2  10   1]
 [ 80   3   8   1]
 [100   3  15   1]
 [120   4  15   1]
 [150   4  20   1]
 [ 90   3  12   1]
 [110   4   7   1]
 [130   5   3   1]] [180 240 260 300 350 280 320 360]


(array([[-1.84530662, -1.73205081, -0.2478408 ,  0.        ],
        [-0.81536804, -0.57735027, -0.64438608,  0.        ],
        [-0.12874232, -0.57735027,  0.7435224 ,  0.        ],
        [ 0.5578834 ,  0.57735027,  0.7435224 ,  0.        ],
        [ 1.58782198,  0.57735027,  1.73488559,  0.        ],
        [-0.47205518, -0.57735027,  0.14870448,  0.        ],
        [ 0.21457054,  0.57735027, -0.84265872,  0.        ],
        [ 0.90119626,  1.73205081, -1.63574927,  0.        ]]),
 array([-1.90113312, -0.82755207, -0.46969171,  0.24602899,  1.14067987,
        -0.11183136,  0.60388935,  1.31961005]))

In [7]:
alpha = 0.1
w = [0, 0, 0, 0]
for i in range(1000):
    w = w - alpha * (X_normalized.T @ X_normalized @ w - X_normalized.T @ y_normalized)
w

array([ 1.2566944 , -0.22622516, -0.39491766,  0.        ])

In [8]:
# 测试数据
X_test = np.array([
    [100, 3, 10, 1],
    [140, 4, 5, 1]
])

# 测试数据的房价
y_test = np.array([290, 395])

# 预测房价
predictions = scaler_X.transform(X_test) @ w
y_pred = scaler_y.inverse_transform(predictions.reshape(-1, 1)).reshape(-1)
print("预测的房价：", y_pred)

mse = mean_squared_error(y_test, y_pred)
print(f"均方误差: {mse}")

预测的房价： [289.97760524 393.70756026]
均方误差: 0.8354510042185225


#### **用sklearn实现**

In [9]:
# 无需显式把截距项添加到特征矩阵中

# 训练数据的三元特征（房屋面积，房间数量，房屋年龄）
X_train = np.array([
    [50, 2, 10],   # 面积50平米，2个房间，10年房龄
    [80, 3, 8],    # 面积80平米，3个房间，8年房龄
    [100, 3, 15],  # 面积100平米，3个房间，15年房龄
    [120, 4, 15],  # 面积120平米，4个房间，15年房龄
    [150, 4, 20],  # 面积150平米，4个房间，20年房龄
    [90, 3, 12],   # 面积90平米，3个房间，12年房龄
    [110, 4, 7],   # 面积110平米，4个房间，7年房龄
    [130, 5, 3]    # 面积130平米，5个房间，3年房龄
])

# 对应的房价（单位：万元）
y_train = np.array([180, 240, 260, 300, 350, 280, 320, 360])

# 测试数据的三元特征（房屋面积，房间数量，房屋年龄）
X_test = np.array([
    [100, 3, 10],  # 100平米，3个房间，10年房龄
    [140, 4, 5]    # 140平米，4个房间，5年房龄
])

# 测试数据的房价（单位：万元）
y_test = np.array([290 , 395])

In [ ]:
# 直接求导法对应模型LinearRegression
from sklearn.linear_model import LinearRegression

# 创建线性回归模型
model_direct = LinearRegression()

# 训练模型
model_direct.fit(X_train, y_train)

# 使用训练好的模型进行预测
y_pred = model_direct.predict(X_test)
'''
LinearRegression参数详解：
fit_intercept: 是否计算截距(默认True)，设为False时数据应已中心化
copy_X: 是否复制X数据(默认True)，设为False可能节省内存但会修改原数据
n_jobs: 并行计算数(默认None)，-1表示使用所有CPU核心
positive: 是否强制系数为正(默认False)，适用于某些特定场景
'''
# 输出预测的房价
print(y_pred)
# 查看斜率（系数）
print("斜率（特征的权重）：", model_direct.coef_)
# 查看截距
print("截距：", model_direct.intercept_)

mse = mean_squared_error(y_test, y_pred)
print(f"均方误差: {mse}")

[289.98331479 393.72080089]
斜率（特征的权重）： [  2.41156841 -14.61067853  -4.37708565]
截距： 136.4293659621802
均方误差: 0.8183143797149522


In [12]:
# 梯度下降法对应SGDRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

# 归一化数据
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_normalized = scaler_X.fit_transform(X_train)
y_normalized = scaler_y.fit_transform(y_train.reshape(-1, 1)).reshape(-1)
# 创建线性回归模型
model = SGDRegressor(eta0=0.1, learning_rate='optimal', penalty=None, max_iter=1000, tol=None, shuffle=False, power_t=0.5)
'''
loss: 损失函数类型(默认"squared_error")，可选"huber"、"epsilon_insensitive"等
penalty: 正则化类型(默认"l2")，可选"l1"、"elasticnet"或None
alpha: 正则化强度(默认0.0001)，值越大惩罚越重
l1_ratio: 弹性网络中L1/L2混合比例(默认0.15)，1表示纯L1，0表示纯L2
fit_intercept: 是否计算截距(默认True)
max_iter: 最大迭代次数(默认1000)
tol: 停止训练的误差容限(默认0.001)，损失变化小于此值则停止
shuffle: 是否每轮迭代打乱数据(默认True)
verbose: 日志详细程度(默认0不输出)
epsilon: Huber/epsilon不敏感损失的阈值(默认0.1)
random_state: 随机种子(默认None)，保证结果可复现
learning_rate: 学习率策略(默认"invscaling")，可选"constant"、"optimal"、"adaptive"
eta0: 初始学习率(默认0.01)
power_t: 反比例衰减指数(默认0.25)，仅当learning_rate="invscaling"时有效
early_stopping: 是否启用早停(默认False)
validation_fraction: 早停验证集比例(默认0.1)，仅当early_stopping=True时有效
n_iter_no_change: 早停等待轮数(默认5)，验证损失连续不下降的epoch数
warm_start: 是否热启动(默认False)，复用上次训练结果作为初始化
average: 是否平均权重(默认False)，可稳定训练结果
'''
# 训练模型
model.fit(X_normalized, y_normalized, coef_init=[0,0,0], intercept_init=[0])

# 使用训练好的模型进行预测
y_pred = model.predict(scaler_X.transform(X_test))
y_pred = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).reshape(-1)

# 输出预测的房价
print(y_pred)
# 查看斜率（系数）
print("斜率（特征的权重）：", model.coef_)
# 查看截距
print("截距：", model.intercept_)

mse = mean_squared_error(y_test, y_pred)
print(f"均方误差: {mse}")

[1.45325651e+13 1.46034898e+14]
斜率（特征的权重）： [1.59369815e+12 1.29474679e+12 1.34221197e+12]
截距： [1.54538514e+12]
均方误差: 1.0768693371000123e+28


### **逻辑回归-二分类**
*虽然名字里有“回归”二字，但逻辑回归用于**分类**而不是回归。逻辑回归从线性回归演化而来，在线性回归中，输出的是一个连续的值，而在逻辑回归中，我们希望输出的是两个类别（例如“0”或“1”）。*

**核心思想**：给二分类的结果赋予置信度，**将线性回归的输出转换为0~1之间的概率**。


#### **Sigmoid函数**

 Sigmoid函数的其公式为：$\sigma(z)={1 \over {1 + e^{-z}}}$


该函数满足上面所提到的所有条件，可以将任意实数$z$​压缩到 (0, 1) 之间，因此可视作为0~1间的概率。

这里的$z$则指的是线性回归的输出，因此逻辑回归的**完整数学表达式**为：

$$y = \sigma(Xw) = {1 \over {1 + e^{-Xw}}}$$



#### **交叉熵损失**

*我们完全可以像线性回归那样对待逻辑回归，用其计算历史数据的均方误差，优化其最小值的参数，训练模型并使用。*

但有一种可以精准反映概率值的损失，也就是我们的**交叉熵损失**，其公式如下：

$$
CrossEntropy = -{1 \over n} \sum \limits _{i=1}^n \left[ y_i \log(\hat{y_i}) + (1 - y_i) \log(1 - \hat{y_i}) \right]
$$

$y_i$ 为样本 $i$ 的真实值，$\hat y_i$ 为样本 $i$ 的预测值。

可以看到，当真实标签为 1 时，该损失为 $-\log(\hat y_i)$ ；当真实标签为 0 时，该损失为 $-\log(1-\hat y_i)$。与真实标签偏离的越远，损失值越大，因此我们的目标仍然是**优化其最小值**。

> 0~1 之间使用均方误差的值普遍偏小，可能会导致浮点数溢出丢失精度

#### 梯度下降法

同线性回归一样，在确立了合理的损失函数后，便可以使用梯度下降法开始学习模型参数。

* 逻辑回归：

  $\mathbf{z} = X\mathbf{w}$

  $\mathbf{\hat y} = \sigma(\mathbf{z}) = {1 \over 1 + e^{-\mathbf{z}}}$

* 损失函数：

  $L = -{1 \over n}\sum\limits_{i=1}^n[y_i\log(\hat y_i) + (1-y_i)\log(1-\hat y_i)]$

梯度下降法的核心是要求出损失函数（我们习惯设其为 $L$）对参数（$w$）的偏导数 $\partial L \over \partial w$：

${\partial L \over \partial w} = {\partial L \over \partial z} \cdot {\partial z \over \partial w} = {1\over n}X^T(\sigma(\mathbf{z})-\mathbf{y}) = {1\over n}X^T({1 \over {1 + e^{-X\mathbf{w}}}}-\mathbf{y})$

有了偏导数后，则可以设置学习率 $\alpha$ 并执行梯度下降算法：

1. 初始化参数 $w$（随机初始化或者全为0）
2. 更新参数：$w = w - \alpha \cdot {\partial L \over \partial w}$
3. 重复更新，直至达到指定次数或收敛

数据集结构
- Feature 1: 年龄 - 申请人的年龄（18-65岁）
- Feature 2: 年收入 - 申请人的年收入（单位：万元）
- Feature 3: 信用评分 - 申请人的信用评分（300-850，分数越高信用越好）
- Feature 4: 不良信用记录 - 申请人是否有不良信用记录（1表示有，0表示没有）
- Label: 信用卡审批结果 - 是否批准信用卡申请（1表示批准，0表示拒绝）

In [4]:
# 自己实现逻辑回归
def logist_regression(X, w):
    z = X @ w
    y = 1 / (1 + np.exp(-z))
    return y

# 定义交叉熵损失的导函数
def dCrossEntropy(X_train, y_train, w):
    n = len(X_train)
    return (1/n) * X_train.T @ (1 / (1+np.exp(-X_train@w)) - y_train)
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 加载数据
data = pd.read_csv('assets/信用卡审批.csv')

# 提取特征和标签
X = data.iloc[:, :-1].values  # 提取前4列作为特征
y = data.iloc[:, -1].values    # 提取最后一列作为标签

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 归一化数据（分类任务不对y进行归一化）
scaler_X = StandardScaler()
X_normalized = scaler_X.fit_transform(X_train)

# 初始化参数
w = np.array([0, 0, 0, 0])

# 梯度下降更新
alpha = 0.1
for i in range(1000):
    df = dCrossEntropy(X_normalized, y_train, w)
    w = w - alpha * df

print(w)

# 预测
y_pred = logist_regression(scaler_X.transform(X_test), w)
y_pred[y_pred >= 0.5] = 1
y_pred[y_pred < 0.5] = 0

print('准确率:', round(accuracy_score(y_test, y_pred), 2))

# 测试示例数据
X_exam = np.array([
    [32, 30, 80, 0],
    [40, 63, 86, 1]
])

print('示例数据结果:', logist_regression(scaler_X.transform(X_exam), w))


[-1.13424594  0.80213428  2.55627524 -0.4769424 ]
准确率: 0.94
示例数据结果: [0.66740043 0.77898745]


In [ ]:
# 使用sklearn逻辑回归模型
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# 创建逻辑回归模型
model = LogisticRegression(max_iter=1000, penalty=None, random_state=42)
'''
penalty: 正则化类型(默认"l2")，可选"l1"、"elasticnet"或None
dual: 是否使用对偶形式(默认False)，仅当solver="liblinear"且penalty="l2"时有效
tol: 优化算法停止阈值(默认0.0001)，损失变化小于此值则停止
C: 正则化强度倒数(默认1)，值越小正则化越强
fit_intercept: 是否计算截距项(默认True)
intercept_scaling: 截距缩放因子(默认1)，仅当solver="liblinear"时有效
class_weight: 类别权重(默认None)，可设为"balanced"或自定义字典
random_state: 随机种子(默认None)，保证结果可复现
solver: 优化算法(默认"lbfgs")，可选"liblinear"、"newton-cg"、"sag"、"saga"等
max_iter: 最大迭代次数(默认100)
multi_class: 多分类策略(默认"auto")，可选"ovr"(一对多)或"multinomial"(多项式)
verbose: 日志详细程度(默认0不输出)
warm_start: 是否热启动(默认False)，复用上次训练结果
n_jobs: 并行计算数(默认None)，-1表示使用所有CPU核心
l1_ratio: 弹性网络混合比例(默认None)，仅当penalty="elasticnet"时有效
'''
# 训练模型
model.fit(X_normalized, y_train)

# 预测测试集
y_pred = model.predict(scaler_X.transform(X_test))

# 计算模型准确率
accuracy = accuracy_score(y_test, y_pred)
print(f"模型准确率: {accuracy:.2f}")

# 测试示例数据
X_exam = np.array([
    [32, 30, 80, 0],
    [40, 63, 86, 1]
])

print('示例数据结果:', model.predict(scaler_X.transform(X_exam)))

模型准确率: 0.94
示例数据结果: [1 1]


### **softmax回归-多分类**

> * Softmax 回归和逻辑回归的关系：Softmax 回归是一般形式的逻辑回归
> * Softmax 回归和神经网络的关系：Softmax 回归也可以看作单层神经网络

#### 算法原理
多分类模型参数就不再是一个向量$\vec{w}$ ，而是一个**矩阵** $W$ （每一列代表一个线性回归模型的参数）：

$W = \begin{bmatrix} w_{11} & w_{12} & \dots & w_{1n} \\ w_{21} & w_{22} & \dots & w_{2n} \\ \vdots & \vdots &  \ddots & \vdots \\ w_{p1} & w_{p2} & \dots & w_{pn} \end{bmatrix}$，这里的 $W$ 表示，在 $n$ 分类的任务中，每个样本有 $p$ 个特征。

此时模型的输出也不再是一个向量 $\vec{z}$，而是一个**矩阵** $Z$（每一行代表一个样本取每个种类的概率）：

$Z = \begin{bmatrix} z_{11} & z_{12} & \dots & z_{1n} \\ z_{21} & z_{22} & \dots & z_{2n} \\ \vdots & \vdots &  \ddots & \vdots \\ z_{m1} & z_{m2} & \dots & z_{mn} \end{bmatrix}$，这里的 $Z$ 表示，总共有 $m$ 个训练样本，每个样本的 $n$ 个类别各自的概率。

即，现在的“线性回归”变成了：$Z = X \cdot W$

#### Softmax 函数

 **Softmax 函数**。它除了具备 Sigmoid 函数所具备的性质外（**04. 逻辑回归**），还能保证每个类别的概率和为1。（Softmax 函数是该模型的核心，也是为什么这个模型叫 Softmax 回归的原因）

Softmax**公式**为：
$$
P(y=j | x_i) = {e^{Z_{ij}} \over \sum_k^n e^{Z_{ik}}}
$$
其中：

* $P(y = j | x_i)$ 表示，第 $i$ 个样本类别为 $j$ 的概率值。
* $Z_{ij}$ 表示第 $i$ 个样本在类别 $j$ 上的线性回归输出值。
* $\sum_k^n Z_{ik}$ 表示第 $i$​ 个样本**所有类别**的线性回归输出值**之和**。

### 交叉熵损失

在逻辑回归中，我们已经学习过了交叉熵损失长什么样子：

$$CrossEntropy = -{1 \over m}\sum\limits_{i=1}^m[y_i\log(\hat y_i) + (1-y_i)\log(1-\hat y_i)]$$

由于当时的输出 $\hat y$ 是一个单值，因此如此使用它。而在现在我们在多分类任务中，每个样本输出的 $\hat y_i$ 是一个向量（向量中的每个值都代表一个类别的概率），因此无法像上面那样去计算。

> 在逻辑回归中我们分析过，交叉熵损失看起来比较复杂，但事实上对每一个类别进行计算的时候只会用到其中的一项，即：当 $y_i$ 为1时，该损失为$-\log(\hat y_i)$；当 $y_i$ 为0时，该损失为$-\log(1-\hat y_i)$。

我们可以参照此法去做等价变换，以简化表示形式。因此多分类任务中，交叉熵损失的公式为：

$$
CrossEntropy = -\frac{1}{m} \sum\limits_{i=1}^{m} \sum\limits_{j=1}^{n} \log(\hat{Y}_{ij}) \quad \text{ \{if } y_j \text{ = ture, else } 0 \}
$$

其意为：**交叉熵损失只会计算在真实类别预测上的损失**（优化该损失函数能同时优化模型在所有类别上的分类，参考 softmax 函数的第 3 个特点）

#### **One-Hot向量**

*一种用于表示分类标签的编码方式，其特点是，向量的长度为标签数，且只有真实标签一个位置为1，其余的位置都为0。*

比如：

- “猫”表为 $[1, 0, 0]$
- “狗”表为 $[0, 1, 0]$
- “鸟”表为 $[0, 0, 1]$

每个类别都用一个长度为 $3$ 的向量来表示，且每个向量中只有一个元素为 $1$，其余元素为 $0$。这样，每个向量之间不存在大小关系的比较，且将标签向量化，很好的避免了上述存在的问题。

在有了One-Hot向量之后，真实标签同样也不在是一个向量 $\mathbf{y}$，而变成了矩阵 $Y$（每一行代表一个样本的One-Hot向量）：

$Y = \begin{bmatrix} y_{11} & y_{12} & \dots & y_{1n} \\ y_{21} & y_{22} & \dots & y_{2n} \\ \vdots & \vdots &  \ddots & \vdots \\ y_{m1} & y_{m2} & \dots & y_{mn} \end{bmatrix}$，这里的 $Y$ 表示，总共有 $m$ 个训练样本，每个样本（每一行）只有一个值为 $1$，其余都为 $0$。

因此，交叉熵损失则变成了：

$$CrossEntropy = -{1 \over m}\sum\limits_{i=1}^m\sum\limits_{j=1}^n Y_{ij} \log(\hat Y_{ij}) \text{ \{if } y_j \text{ = ture, else } 0 \}= -{1 \over m}\sum\limits_{i=1}^m Y * \log(\hat Y)$$

* " $*$ ” 表示矩阵对应元素相乘，而不是矩阵乘法（同`numpy`里的写法）。


In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 加载数据
iris = load_iris()
X, y = iris.data, iris.target

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 创建Softmax回归模型
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
'''
penalty: 正则化类型(默认"l2")，可选"l1"、"elasticnet"或None
dual: 是否使用对偶形式(默认False)，仅当solver="liblinear"且penalty="l2"时有效
tol: 优化算法停止阈值(默认0.0001)，损失变化小于此值则停止
C: 正则化强度倒数(默认1)，值越小正则化越强
fit_intercept: 是否计算截距项(默认True)
intercept_scaling: 截距缩放因子(默认1)，仅当solver="liblinear"时有效
class_weight: 类别权重(默认None)，可设为"balanced"或自定义字典
random_state: 随机种子(默认None)，保证结果可复现
solver: 优化算法(默认"lbfgs")，可选"liblinear"、"newton-cg"、"sag"、"saga"等
max_iter: 最大迭代次数(默认100)
multi_class: 多分类策略(默认"auto")，可选"ovr"(一对多)或"multinomial"(多项式)
verbose: 日志详细程度(默认0不输出)
warm_start: 是否热启动(默认False)，复用上次训练结果
n_jobs: 并行计算数(默认None)，-1表示使用所有CPU核心
l1_ratio: 弹性网络混合比例(默认None)，仅当penalty="elasticnet"时有效
'''
# 训练模型
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)

print('准确率:', round(accuracy_score(y_test, y_pred), 2))

# 测试示例数据
X_exam = np.array([
    [4.8, 3.0, 1.1, 0.1]
])

y_exam = model.predict(X_exam)[0]
print('示例数据结果:', iris.target_names[y_exam])

准确率: 1.0
示例数据结果: setosa
